# PLTR vs NVDA: Genuine Correlation or Shared QQQ Beta?

This notebook investigates whether the recent co-movement between Palantir (PLTR) and NVIDIA (NVDA) reflects genuine pairwise correlation or is simply shared exposure to the Nasdaq-100 (QQQ). The key technique: regress both stocks on QQQ to strip out the market factor, then check if the **residuals** are still correlated.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from correlation.analysis import (
    compute_rolling_correlation,
    compute_rolling_beta,
    compute_residuals,
    compute_rolling_residual_correlation,
    compute_residual_stats,
)

plt.style.use("dark_background")
plt.rcParams["figure.dpi"] = 120

## 1. Data Loading

Pull 1 year of daily close prices for PLTR, NVDA, and QQQ via yfinance, then compute daily percentage returns.

In [ ]:
tickers = ["PLTR", "NVDA", "QQQ"]
data = yf.download(tickers, period="1y")["Close"]
returns = data.pct_change().dropna()

print(f"Date range: {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Trading days: {len(returns)}")
returns.tail()

## 2. Rolling Pairwise Correlation

21-day and 63-day rolling Pearson correlation between PLTR and NVDA daily returns. **What to look for:** if both lines stay elevated (> 0.5), the two stocks are consistently moving together. The 63-day line is smoother and shows the structural trend; the 21-day line captures short-term regime shifts.

In [ ]:
corr_21 = compute_rolling_correlation(returns["PLTR"], returns["NVDA"], 21)
corr_63 = compute_rolling_correlation(returns["PLTR"], returns["NVDA"], 63)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(corr_21.index, corr_21, label="21-day", alpha=0.8, linewidth=1)
ax.plot(corr_63.index, corr_63, label="63-day", alpha=0.9, linewidth=2)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax.set_title("Rolling Pairwise Correlation: PLTR vs NVDA", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Pearson Correlation")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Rolling Beta to QQQ

63-day rolling OLS beta of PLTR and NVDA to QQQ. **What to look for:** if both betas are converging to similar values, they share similar market exposure. High betas (> 1) indicate amplified sensitivity to QQQ moves. If the betas differ significantly, any raw correlation is less likely to be *just* shared beta.

In [ ]:
beta_pltr = compute_rolling_beta(returns["PLTR"], returns["QQQ"], 63)
beta_nvda = compute_rolling_beta(returns["NVDA"], returns["QQQ"], 63)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(beta_pltr.index, beta_pltr, label="PLTR beta", linewidth=2)
ax.plot(beta_nvda.index, beta_nvda, label="NVDA beta", linewidth=2)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.5, label="Beta = 1")
ax.set_title("63-Day Rolling Beta to QQQ", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("OLS Beta")
ax.legend()
plt.tight_layout()
plt.show()